# 🚀 Run the Streamlit RAG App from Colab (free)
This notebook writes `app.py`, launches Streamlit inside Colab, and gives you a public link via localtunnel. 

Use it to **demo the app without GitHub**. For a permanent link, deploy from GitHub to Streamlit Community Cloud instead (see the guide).

## 1. Install dependencies

In [ ]:
!pip install -q streamlit pypdf sentence-transformers transformers torch numpy
!npm install -q localtunnel

## 2. Write `app.py`

In [ ]:
%%writefile app.py
"""Streamlit front end for the free PDF RAG lab.
Run locally:  streamlit run app.py
"""
import numpy as np
import streamlit as st
import torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "google/flan-t5-base"   # small enough for Streamlit Community Cloud's free tier

st.set_page_config(page_title="PDF RAG Chat", page_icon="📄")
st.title("📄 Chat with your PDF (free RAG)")
st.caption("Open-source models running locally: no API key, no token limits.")


@st.cache_resource(show_spinner="Loading models (first run takes a minute)...")
def load_models():
    embedder = SentenceTransformer(EMBED_MODEL, device="cpu")
    tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
    llm = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL)
    return embedder, tokenizer, llm


def read_pdf(file):
    reader = PdfReader(file)
    return "\n".join((p.extract_text() or "") for p in reader.pages)


def chunk_text(text, chunk_words=200, overlap=40):
    words = text.split()
    step = chunk_words - overlap
    chunks = [" ".join(words[i:i + chunk_words]) for i in range(0, len(words), step)]
    return [c for c in chunks if c]


def retrieve(question, chunks, embeddings, embedder, k):
    q = embedder.encode([question], normalize_embeddings=True)[0]
    top = np.argsort(embeddings @ q)[::-1][:k]
    return [chunks[i] for i in top]


def answer(question, chunks, embeddings, models, k):
    embedder, tokenizer, llm = models
    context = "\n\n".join(retrieve(question, chunks, embeddings, embedder, k))
    prompt = (
        "Answer the question using only the context below. "
        "If the answer is not in the context, say \"I don't know\".\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=256, num_beams=4)
    return tokenizer.decode(out[0], skip_special_tokens=True), context


# ---- Sidebar: upload + settings ---------------------------------------------
with st.sidebar:
    st.header("1. Upload")
    pdf = st.file_uploader("Upload a PDF", type="pdf")
    k = st.slider("Chunks to retrieve", 1, 6, 3)
    if st.button("Clear chat"):
        st.session_state.history = []

models = load_models()

# ---- Build the index once per uploaded file ---------------------------------
if pdf is not None and st.session_state.get("file_name") != pdf.name:
    with st.spinner("Reading and indexing your PDF..."):
        text = read_pdf(pdf)
        if not text.strip():
            st.error("No text found. The PDF may be a scanned image (needs OCR).")
            st.stop()
        chunks = chunk_text(text)
        embeddings = models[0].encode(chunks, normalize_embeddings=True)
        st.session_state.update(file_name=pdf.name, chunks=chunks,
                                embeddings=embeddings, history=[])
    st.sidebar.success(f"Indexed {len(chunks)} chunks from {pdf.name}")

# ---- Chat -------------------------------------------------------------------
if "chunks" not in st.session_state:
    st.info("👈 Upload a PDF in the sidebar to get started.")
    st.stop()

for role, msg in st.session_state.get("history", []):
    st.chat_message(role).write(msg)

if question := st.chat_input("Ask something about your PDF"):
    st.chat_message("user").write(question)
    with st.spinner("Thinking..."):
        reply, ctx = answer(question, st.session_state.chunks,
                            st.session_state.embeddings, models, k)
    st.chat_message("assistant").write(reply)
    with st.expander("Retrieved context"):
        st.write(ctx)
    st.session_state.history.append(("user", question))
    st.session_state.history.append(("assistant", reply))

## 3. Launch Streamlit

In [ ]:
!streamlit run app.py --server.port 8501 --server.headless true &>/content/logs.txt &

## 4. Get the public link
1. Run the cell below. It prints your **tunnel password** (a public IP address).
2. Click the `https://....loca.lt` link, paste the password on the page, and click Submit.
3. Keep this cell running while you use the app.

In [ ]:
import urllib.request, time
time.sleep(5)
print("Tunnel password:", urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip())
!npx localtunnel --port 8501